## Assignment 2: Code Whiteboard Tutor

**Assignment 2 Group 12 :**
- Chung-Ping Chen : 7008394
- Chun-Yu, Wu : 92302916
- Shih-Yun Huang : 20645816
- Kao Ching : 85768427

Main goal: Use a multimodal Large Language Model (LLM) to build a UI application that allows users to upload a photo of their handwritten Python code and receive suggestions for code improvements.

The main functionalities that must be included are:
- Transcribing handwritten Python code into a code snippet
- Running static analysis and explaining bugs (if any) in natural language
- Suggesting bug fixes, improvements, or efficiency tweaks to the code snippet


In [7]:
# Install required packages
# %pip -q install gradio unstructured sentence-transformers "google-genai>=1.50"
%pip install --no-cache-dir gradio pillow "google-genai>=1.50"

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Import necessary libraries
import os
import time
from PIL import Image

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata


# For Google Colab, use the secret tab (with key symbol) on the left navigation bar
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.5-flash-lite"

In [ ]:
# For a local environment, put your Gemini API key here instead of using Colab secrets
from google import genai
from google.genai import types

api_key = ''  # TODO: put your Gemini API key
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.5-flash-lite"

### ⚠️ A Note on Rate Limits

We're using `gemini-3.5-flash-lite` because it's a free-tier model, but rate limits still apply (and vary by model).

See the [Rate Limits docs](https://ai.google.dev/gemini-api/docs/rate-limits) for details. Three things to watch:

- **RPM (Requests Per Minute):** API calls allowed per 60-second window.
- **TPM (Tokens Per Minute):** input tokens allowed per 60-second window.
- **RPD (Requests Per Day):** total requests allowed per day (resets at midnight Pacific time).

Exceeding any one triggers a `429 RESOURCE_EXHAUSTED` error. If that happens, wait a few seconds and try again, and don't send too many requests in a short time.

In [16]:
import gradio as gr
# Add any additional imports required for your implementation

"""
Note: This template serves as a starting point.
You are expected to modify and extend the code to fulfill the assignment requirements.
"""


def transcribe_image_code(image_file: Image.Image) -> str:
    """
    Step 1: Transcribes code from the input image using a multimodal LLM.

    Parameters:
        image_file (Image.Image): The uploaded image containing handwritten code.

    Returns:
        str: The transcribed code as a string.
    """

    if image_file is None:
        raise gr.Error("Please upload an image of handwritten Python code.")

    prompt = (
        "Transcribe the handwritten Python code in this image exactly as written. "
        "Preserve indentation, identifiers, comments, and syntax, including mistakes. "
        "Do not fix, explain, or complete the code. "
        "Return only the transcribed source code with no markdown fences or extra text."
    )

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=[image_file, prompt],
        )
        return _parse_transcribe_response((response.text or "").strip())
    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Error during transcription: {str(e)}")

def _parse_transcribe_response(transcribed_code: str) -> str:
    transcribed_code = _strip_code_fences(transcribed_code)
    
    if transcribed_code.lower().startswith("python"):
        transcribed_code = transcribed_code[6:].strip()

    if not transcribed_code:
        raise gr.Error("Could not transcribe any code from the image.")
    return transcribed_code


def _strip_code_fences(text: str) -> str:
    cleaned = (text or "").strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()
    return cleaned



def analyze_code(code_block: str) -> str:
    """Analyze Python code and return a Markdown report."""
    if not code_block or not code_block.strip():
        raise gr.Error("Please provide a valid code snippet.")

    prompt = f"""You are an expert Python developer and code reviewer.

Analyze the following Python code.

Your tasks:
1. Identify syntax errors, logic bugs, and edge-case issues.
2. Explain why each issue occurs and how to fix it.
3. Suggest readability or performance improvements when relevant.
4. Do not generate the complete corrected code.

Return a Markdown report with these sections:

## Bugs and Issues
For each issue, include:
- Location
- Description
- Suggested Fix

## Improvement Suggestions
List optional readability and efficiency improvements.
Clearly distinguish optional improvements from required bug fixes.

If no bugs are found, explicitly state that.

Code to analyze:
```python
{code_block}
```
"""

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=[prompt],
        )

        analysis = (response.text or "").strip()
        if not analysis:
            raise gr.Error("The model returned an empty analysis.")

        return analysis

    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Error during analysis: {e}")


def refine_code(code_block: str, analysis: str) -> str:
    """Refine Python code using the original code and analysis."""
    if not code_block or not analysis:
        raise gr.Error("Code and analysis are required.")

    prompt = f"""You are an expert Python developer.

Refine the original Python code based on the provided analysis.

Requirements:
1. Fix all confirmed functional bugs identified in the analysis.
2. Preserve the original function signatures and intended behavior.
3. Apply safe readability and efficiency improvements when useful.
4. Do not introduce unrelated changes.
5. Return the complete corrected Python source code.
6. Output Python code only, without explanations or Markdown fences.

Original code:
```python
{code_block}
```

Analysis:
{analysis}
"""

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=[prompt],
        )

        refined_code = _strip_code_fences(
            (response.text or "").strip()
        )

        if not refined_code:
            raise gr.Error("The model returned empty refined code.")

        return refined_code

    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(f"Error during code refinement: {e}")



## Implement the Gradio Interface to run the app
def process_image(image, progress=gr.Progress()):
    if image is None:
        raise gr.Error("Please upload an image.")

    code = ""
    analysis = ""
    refined_code = ""

    try:
        # Step 1: OCR
        progress(0, desc="Transcribing handwritten code...")
        yield "🔄 Step 1/3: Transcribing code...", code, analysis, refined_code

        code = transcribe_image_code(image)
        progress(1 / 3)
        yield "✅ Transcription completed.", code, analysis, refined_code

        # Step 2: Analysis
        progress(1 / 3, desc="Analyzing code...")
        yield "🔄 Step 2/3: Analyzing code...", code, analysis, refined_code

        analysis = analyze_code(code)
        progress(2 / 3)
        yield "✅ Analysis completed.", code, analysis, refined_code

        # Step 3: Refinement
        progress(3 / 3, desc="Refining code...")
        yield "🔄 Step 3/3: Refining code...", code, analysis, refined_code

        refined_code = refine_code(code, analysis)
        progress(1)
        yield "✅ All steps completed!", code, analysis, refined_code

    except Exception as e:
        yield f"❌ Processing failed: {e}", code, analysis, refined_code

with gr.Blocks(title="Handwritten Python Code Debugger") as app:
    gr.Markdown(
        """
        # 📝 Handwritten Python Code Debugger

        Upload an image of handwritten Python code to automatically
        transcribe, analyze, and debug it.
        """
    )

    status = gr.Markdown("**Status:** Waiting for an image...")

    with gr.Row():
        # Left: Image input and transcription
        with gr.Column(scale=1):
            image_input = gr.Image(
                type="pil",
                label="Upload Handwritten Code",
                sources=["upload"],
            )

            code_editor = gr.Code(
                label="Transcribed Code (Editable)",
                language="python",
                interactive=True,
            )

            refined_code_output = gr.Code(
                label="Corrected Python Code",
                language="python",
                interactive=False,
            )

        # Right: Analysis and corrected code
        with gr.Column(scale=1):
            analysis_output = gr.Markdown(
                label="Bug Analysis and Recommendations",
                value="*Analysis results will appear here.*",
                container=True,
            )



    # Automatically start the pipeline on upload
    image_input.upload(
        fn=process_image,
        inputs=[image_input],
        outputs=[
            status,
            code_editor,
            analysis_output,
            refined_code_output,
        ],
        show_progress="full",
    )

    # Allow users to rerun analysis after editing
    analyze_btn = gr.Button(
        "Reanalyze Edited Code",
        variant="secondary",
    )

    def reanalyze_code(code, progress=gr.Progress()):
        if not code or not code.strip():
            yield "⚠️ No code to analyze.", "", ""
            return

        progress(0, desc="Analyzing edited code...")
        yield "🔄 Analyzing edited code...", "", ""

        try:
            analysis, refined_code = analyze_code(code)
            progress(1.0, desc="Analysis completed")
            yield "✅ Analysis completed!", analysis, refined_code
        except Exception as e:
            yield f"❌ Analysis failed: {e}", "", ""

    analyze_btn.click(
        fn=reanalyze_code,
        inputs=[code_editor],
        outputs=[
            status,
            analysis_output,
            refined_code_output,
        ],
    )


if __name__ == "__main__":
    app.queue().launch()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


### Additional Resource: Running test cases on LLM-generated code.

In [ ]:
"""
Example of running test cases on LLM-generated code.
You do not need to follow this exact implementation for your code."""

import json
import importlib.util

def run_tests(filename_original, function_name, json_test_path):
    # Load test cases
    with open(json_test_path, 'r') as f:
        test_cases = json.load(f)["test_case"]

    # Load the function from the file
    spec = importlib.util.spec_from_file_location(function_name, filename_original)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    func = getattr(module, function_name)

    # Run tests
    for idx, case in enumerate(test_cases):
        try:
            inputs = case["input"]
            expected = case["expected"]
            result = func(*inputs) if isinstance(inputs, (list, tuple)) else func(inputs)
            assert result == expected, f"input={inputs}, expected={expected}, got={result}"
            print(f"Test {idx+1} passed.")
        except Exception as e:
            print(f"Test {idx+1} failed: {e}")


In [ ]:
"""
Make sure to save the final code (after transcribing and performing static analysis) into a Python file.
For example, if you have saved the final code transcribed and fixed by the LLM as example_llm_code.py,
you can run the test cases using the format below.

You can also add more inputs and expected outputs to the JSON file to run additional tests.
It is encouraged to add more test cases to ensure the robustness of your code.
"""

run_tests(
    filename_original="example_llm_code.py",
    function_name="bucketsort",
    json_test_path="test_case_bucketsort.json"
)
